# AdFatigueRadar — NLP & Model Exploration Notebook
**Topic #15: AdTech & Marketing Analytics | CODEBEGUN HACKZEN 2026**

This interactive notebook provides step-by-step exploration, validation, and benchmarking of the **Person 1 AI / NLP Layer** for AdFatigueRadar:
1. **Two-Stage NLP Pipeline**: General Sentiment + 8-Class Ad-Fatigue Taxonomy.
2. **Disambiguation Rules**: Repeated ad exposure (`fatigue`) vs Repeat purchase satisfaction (`positive`).
3. **Mockery vs Banter Separation**: Filtering out non-harmful meme slang ($w=0.0$) while catching campaign roasts ($w=0.75$).
4. **Critical Complaint Detection**: Gated by calibrated confidence ($\ge 0.85$) for product/service defects.
5. **Baseline Benchmarking**: Naive Sentiment-Only (Baseline C) vs Combined Taxonomy Guard.
6. **Latency Profiling**: Demonstrating sub-millisecond CPU inference and cache acceleration.

In [ ]:
import os
import sys
import json
import pandas as pd

# Add parent directory to path
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

from backend.nlp import CommentEvent, NLPResult, TAXONOMY_CATEGORIES
from backend.nlp.classifier import CommentClassifier
from backend.nlp.calibration import TemperatureScaler
from backend.nlp.baselines import SentimentOnlyBaseline
from backend.nlp.benchmark import NLPLatencyBenchmark

print("AdFatigueRadar NLP modules successfully imported!")
print("Available Taxonomy Categories:", TAXONOMY_CATEGORIES)

## 1. Dataset Inspection & Class Distribution
Let's inspect both the training corpus and the 320 human-audited held-out test samples.

In [ ]:
train_path = os.path.join(BASE_DIR, "data", "training", "labels.jsonl")
test_path = os.path.join(BASE_DIR, "data", "test_human_audited", "labels.jsonl")

df_train = pd.read_json(train_path, lines=True)
df_test = pd.read_json(test_path, lines=True)

print(f"Training Set: {len(df_train)} samples")
print(f"Held-Out Human Audited Test Set: {len(df_test)} samples")

print("\n--- Test Set Class Breakdown ---")
print(df_test["category"].value_counts())

## 2. Interactive Classifier Demonstration
Testing the core disambiguation rules and critical complaint identification.

In [ ]:
classifier = CommentClassifier(use_cache=False)

demo_comments = [
    "Bro this ad again 😂 I see it on my feed every 5 minutes",
    "Bought again, absolute best purchase I made this year!",
    "Ordered again, thank you for the wonderful customer support!",
    "The product broke on day 2, complete waste of money cheap plastic defect",
    "Ordered 3 weeks ago and support refuses to issue a refund or reply",
    "The acting in this commercial is killing me 💀 who approved this clown show 🤡",
    "Bro got that unspoken rizz let him cook 🔥",
    "Click link in bio for free crypto telegram signals",
    "How much is shipping to Canada for two units?",
    "Bro this price is broken, such an amazing deal bought again!"
]

results = []
for text in demo_comments:
    res = classifier.classify_text(text)
    results.append({
        "Comment Text": text,
        "Predicted Category": res.category,
        "Sentiment": res.sentiment,
        "Confidence": f"{res.confidence:.2%}",
        "Critical Complaint?": res.critical_complaint
    })

pd.DataFrame(results)

## 3. Naive Sentiment Baseline vs Combined Taxonomy Guard
Benchmarking how Baseline C (naive sentiment) mistakenly alarms during viral banter storms, whereas the combined guard isolates banter ($w=0.0$).

In [ ]:
baseline_evaluator = SentimentOnlyBaseline(negative_threshold=0.30)

banter_feed = [
    CommentEvent(
        event_id=f"banter_{i}",
        timestamp="2026-09-30T12:00:00Z",
        campaign_id="camp_01",
        ad_id="ad_01",
        author_id=f"user_{i}",
        text="The acting in this ad is killing me 💀 bro really thought he cooked" if i % 2 == 0 else "Bro got that unspoken rizz let him cook 🔥"
    )
    for i in range(20)
]

metrics = baseline_evaluator.evaluate_window(banter_feed)
print("--- Naive Sentiment-Only Evaluation ---")
print(f"Total Comments Evaluated : {metrics.total_comments}")
print(f"Raw Negative Share       : {metrics.raw_negative_share:.2%}")
print(f"Alert Triggered?         : {metrics.alert_triggered}")
print(f"Is False Alarm on Banter?: {metrics.is_false_alarm_on_banter}")

## 4. Latency & Throughput Benchmark
Live profiling without replay caching to measure true single-comment and batched CPU execution speed.

In [ ]:
benchmark = NLPLatencyBenchmark(classifier=classifier)
report = benchmark.run_benchmark(num_samples=100)

print(f"Throughput               : {report.throughput_comments_per_sec:.2f} comments/second")
print(f"Latency p50 (Median)     : {report.latency_p50_ms:.2f} ms")
print(f"Latency p95              : {report.latency_p95_ms:.2f} ms")
print(f"Latency Max              : {report.latency_max_ms:.2f} ms")
print(f"Sub-Minute Latency Passed: {report.sub_minute_target_passed}")